In [5]:
import os
import numpy as np
import pandas as pd
from scipy.spatial import Delaunay
from pathlib import Path
from tqdm import tqdm
import glob
import torch
from torch_geometric.data import Data
import geopandas as gpd

In [ ]:
# Transcript coordinates
df_qv20 = pd.read_parquet('filtered_df.parquet')
df = df_qv20[df_qv20['qv'] >= 30]
coords = df[['x', 'y']].to_numpy()

In [ ]:
def compute_delaunay_edges(coords_tile, global_indices, coords_full, distance_thresh=1.0):
    tri = Delaunay(coords_tile)
    edge_set = set()
    for simplex in tri.simplices:
        for i, j in [(0, 1), (1, 2), (2, 0)]:
            u, v = simplex[i], simplex[j]
            if u == v:
                continue
            u_glob, v_glob = global_indices[u], global_indices[v]
            edge = (min(u_glob, v_glob), max(u_glob, v_glob))
            edge_set.add(edge)

    edges = np.array(list(edge_set), dtype=np.int64)
    dist = np.linalg.norm(coords_full[edges[:, 0]] - coords_full[edges[:, 1]], axis=1)
    keep = dist <= distance_thresh
    return edges[keep]

def delaunay_tiling_to_parquet(coords, tile_size=512, stride=480, distance_thresh=1.0, out_dir="tile_edges"):
    Path(out_dir).mkdir(exist_ok=True)
    x_min, x_max = coords[:, 0].min(), coords[:, 0].max()
    y_min, y_max = coords[:, 1].min(), coords[:, 1].max()

    tile_id = 0
    for x0 in tqdm(np.arange(x_min, x_max, stride), desc="Tiling x"):
        for y0 in np.arange(y_min, y_max, stride):
            x1, y1 = x0 + tile_size, y0 + tile_size
            tile_mask = ((coords[:, 0] >= x0) & (coords[:, 0] <= x1) &
                         (coords[:, 1] >= y0) & (coords[:, 1] <= y1))
            if tile_mask.sum() < 10:
                continue

            coords_tile = coords[tile_mask]
            global_idx = np.where(tile_mask)[0]
            tile_edges = compute_delaunay_edges(coords_tile, global_idx, coords, distance_thresh=distance_thresh)

            df_tile = pd.DataFrame(tile_edges, columns=["i", "j"])
            df_tile.to_parquet(os.path.join(out_dir, f"tile_{tile_id:05d}.parquet"), index=False)
            tile_id += 1

def merge_and_filter_edges(coords, edge_dir="tile_edges", distance_thresh=1.0):
    edge_files = list(Path(edge_dir).glob("*.parquet"))
    all_dfs = [pd.read_parquet(p) for p in edge_files]
    edges_df = pd.concat(all_dfs, ignore_index=True)

    # Sort and dedup
    edges_df[['i', 'j']] = np.sort(edges_df[['i', 'j']].values, axis=1)
    edges_df = edges_df.drop_duplicates().reset_index(drop=True)

    # Filter by physical distance again, post-merge
    coords_tensor = coords[edges_df[['i', 'j']].values]
    dists = np.linalg.norm(coords_tensor[:, 0] - coords_tensor[:, 1], axis=1)
    edges_df = edges_df[dists <= distance_thresh].reset_index(drop=True)

    return edges_df

In [ ]:
delaunay_tiling_to_parquet(
    coords=coords, 
    tile_size=512, 
    stride=480, 
    distance_thresh=1.0, 
    out_dir="tile_edges"
)
#
edges_df = merge_and_filter_edges(
    coords=coords, 
    edge_dir="tile_edges", 
    distance_thresh=1.0  
)

In [ ]:
# Load and merge edge Parquet
all_dfs = [pd.read_parquet(p) for p in glob.glob("tile_edges/*.parquet")]
all_edges_df = pd.concat(all_dfs, ignore_index=True)

# Drop symmetric duplicates
all_edges_df[['i', 'j']] = np.sort(all_edges_df[['i', 'j']].values, axis=1)
all_edges_df = all_edges_df.drop_duplicates().reset_index(drop=True)

# Convert to edge_index
e_i = all_edges_df['i'].to_numpy()
e_j = all_edges_df['j'].to_numpy()

# Build symmetric edge index
edge_index_np = np.vstack([e_i, e_j])
edge_index_np = np.concatenate([edge_index_np, edge_index_np[::-1]], axis=1)  # [2, 2*E]
edge_index = torch.tensor(edge_index_np, dtype=torch.long)

# Load transcript-level gene embeddings as features
gene_embed = torch.load('gene_embedding_dict.pt')  # dict: gene_name -> 20D array/list

x = torch.stack(
    [torch.tensor(gene_embed.get(g, np.zeros(20)), dtype=torch.float32) for g in df['feature_name'].values],
    dim=0
)  # shape: (N, 20)

# Build PyG Data object
data = Data(x=x, edge_index=edge_index)

# Add coordinates and feature names 
data.pos = torch.tensor(df[['x', 'y']].values, dtype=torch.float32)
data.gene_name = df['feature_name'].values.tolist()

torch.save(data, 'transcript_graph_data.pt')

/var/folders/_2/50fjjk3d1696gqy3jz2ypb9j88m1t6/T/ipykernel_64526/2539187862.py:27: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  [torch.tensor(gene_embed.get(g, np.zeros(20)), dtype=torch.float32) for g in df['feature_name'].values],
